# Pipeline de prédiction du prix des maisons sur Snowflake

**Data Engineering + Machine Learning + Model Registry + Streamlit**  
Workshop

Ce notebook reprend ton pipeline en séparant proprement les étapes en cellules SQL et Python.


## 1. Configuration de l'environnement

In [ ]:
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.ML;

USE DATABASE HOUSE_PRICE_DB;
USE SCHEMA ML;

CREATE WAREHOUSE IF NOT EXISTS ML_WH
  WAREHOUSE_SIZE = 'X-SMALL'
  AUTO_SUSPEND   = 60
  AUTO_RESUME    = TRUE;

USE WAREHOUSE ML_WH;

## 2. Vérification des dépendances Python

In [ ]:
import pkg_resources

deps = ["scikit-learn", "pandas", "numpy", "xgboost", "matplotlib", "seaborn"]

print("Vérification des dépendances Python :")
print("─" * 40)
for d in deps:
    try:
        v = pkg_resources.get_distribution(d).version
        print(f"  OK   {d:<20} version {v}")
    except Exception:
        print(f"  MANQUANT  {d}")
print("─" * 40)
print("Verification terminee.")

## 3. Ingestion des données depuis S3

In [ ]:

CREATE OR REPLACE FILE FORMAT JSON_FORMAT
  TYPE = 'JSON'
  STRIP_OUTER_ARRAY = TRUE;

CREATE OR REPLACE STAGE HOUSE_PRICE_STAGE
  URL = 's3://logbrain-datalake/datasets/house_price/'
  FILE_FORMAT = JSON_FORMAT;

LIST @HOUSE_PRICE_STAGE;

CREATE OR REPLACE TABLE HOUSE_PRICE_RAW (
  PRICE             NUMBER,
  AREA              NUMBER,
  BEDROOMS          NUMBER,
  BATHROOMS         NUMBER,
  STORIES           NUMBER,
  MAINROAD          VARCHAR,
  GUESTROOM         VARCHAR,
  BASEMENT          VARCHAR,
  HOTWATERHEATING   VARCHAR,
  AIRCONDITIONING   VARCHAR,
  PARKING           NUMBER,
  PREFAREA          VARCHAR,
  FURNISHINGSTATUS  VARCHAR
);

COPY INTO HOUSE_PRICE_RAW (
  PRICE, AREA, BEDROOMS, BATHROOMS, STORIES,
  MAINROAD, GUESTROOM, BASEMENT, HOTWATERHEATING,
  AIRCONDITIONING, PARKING, PREFAREA, FURNISHINGSTATUS
)
FROM (
  SELECT
    $1:price::NUMBER,
    $1:area::NUMBER,
    $1:bedrooms::NUMBER,
    $1:bathrooms::NUMBER,
    $1:stories::NUMBER,
    $1:mainroad::VARCHAR,
    $1:guestroom::VARCHAR,
    $1:basement::VARCHAR,
    $1:hotwaterheating::VARCHAR,
    $1:airconditioning::VARCHAR,
    $1:parking::NUMBER,
    $1:prefarea::VARCHAR,
    $1:furnishingstatus::VARCHAR
  FROM @HOUSE_PRICE_STAGE/Housing_Price_Data.json
  (FILE_FORMAT => JSON_FORMAT)
)
ON_ERROR = 'CONTINUE';

SELECT COUNT(*) FROM HOUSE_PRICE_RAW;
SELECT * FROM HOUSE_PRICE_RAW LIMIT 5;

## 4. Exploration des données avec Snowpark

In [ ]:
# ── Exploration et nettoyage des données avec Snowpark ─────────────────────

from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, sum as sum_
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Session Snowflake
session = get_active_session()

# Chargement de la table
df = session.table("HOUSE_PRICE_RAW")

# ── Aperçu initial ─────────────────────────────────────────────────────────
rows_before = df.count()
print(f"Nombre de lignes initial : {rows_before}")
df.show(5)

# ── Suppression des doublons ───────────────────────────────────────────────
df = df.drop_duplicates()

rows_after = df.count()
print(f"\nNombre de lignes après suppression des doublons : {rows_after}")
print(f"Nombre de doublons supprimés : {rows_before - rows_after}")

# ── Schéma et statistiques ─────────────────────────────────────────────────
print("\nSchéma des données :")
df.printSchema()

print("\nStatistiques descriptives :")
df.describe().show()

# ── Valeurs manquantes ─────────────────────────────────────────────────────
print("\nNombre de valeurs nulles par colonne :")

null_counts = df.select([
    sum_(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
null_counts.show()

# ── Définition des variables ───────────────────────────────────────────────
print("\nVariable cible : PRICE")
print("Variables explicatives :")

FEATURES = [
    "AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING",
    "MAINROAD", "GUESTROOM", "BASEMENT",
    "HOTWATERHEATING", "AIRCONDITIONING", "PREFAREA", "FURNISHINGSTATUS"
]

for f in FEATURES:
    print(f"  - {f}")

## 5. Exploration visuelle et corrélations

In [ ]:
df_pd = df.to_pandas()

df_corr = df_pd.copy()
binary_cols = [
    'MAINROAD', 'GUESTROOM', 'BASEMENT',
    'HOTWATERHEATING', 'AIRCONDITIONING', 'PREFAREA'
]
for c in binary_cols:
    df_corr[c] = df_corr[c].map({'yes': 1, 'no': 0})

df_corr['FURNISHINGSTATUS'] = df_corr['FURNISHINGSTATUS'].map(
    {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
)

plt.figure(figsize=(10, 7))
sns.heatmap(df_corr.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matrice de corrélation — House Price Dataset")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(df_pd['PRICE'], bins=40, kde=True, color="#4C72B0")
plt.title("Distribution des prix de vente")
plt.xlabel("Prix")
plt.ylabel("Fréquence")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(data=df_pd, x='FURNISHINGSTATUS', y='PRICE', palette="coolwarm")
plt.title("Prix selon l'état d'ameublement")
plt.tight_layout()
plt.show()

## 6. Feature engineering et sauvegarde

In [ ]:
# ── Feature Engineering avec Snowpark ──────────────────────────────────────

from snowflake.snowpark.functions import col, when

# Mapping binaire générique
def encode_binary(column_name):
    return when(col(column_name) == "yes", 1).otherwise(0)

# Feature engineering
df_clean = (
    df
    .with_column("MAINROAD_ENC",        encode_binary("MAINROAD"))
    .with_column("GUESTROOM_ENC",       encode_binary("GUESTROOM"))
    .with_column("BASEMENT_ENC",        encode_binary("BASEMENT"))
    .with_column("HOTWATER_ENC",        encode_binary("HOTWATERHEATING"))
    .with_column("AIRCON_ENC",          encode_binary("AIRCONDITIONING"))
    .with_column("PREFAREA_ENC",        encode_binary("PREFAREA"))
    .with_column(
        "FURNISHING_ENC",
        when(col("FURNISHINGSTATUS") == "furnished", 2)
        .when(col("FURNISHINGSTATUS") == "semi-furnished", 1)
        .otherwise(0)
    )
)
df_clean.select(
    col("MAINROAD"),
    col("MAINROAD_ENC")
).show(5)
df_clean.group_by("FURNISHING_ENC").count().show()
# Sauvegarde dans Snowflake
df_clean.write.mode("overwrite").save_as_table("HOUSE_PRICE_FEATURES")

print("Table HOUSE_PRICE_FEATURES créée et sauvegardée avec succès.")

## 7. Préparation des données pour le Machine Learning

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_feat = session.table("HOUSE_PRICE_FEATURES").to_pandas()

FEATURES = [
    "AREA", "BEDROOMS", "BATHROOMS", "STORIES", "PARKING",
    "MAINROAD_ENC", "GUESTROOM_ENC", "BASEMENT_ENC",
    "HOTWATER_ENC", "AIRCON_ENC", "PREFAREA_ENC", "FURNISHING_ENC"
]
TARGET = "PRICE"

X = df_feat[FEATURES]
y = df_feat[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")

# Binning construit à partir du train uniquement
q33 = y_train.quantile(0.33)
q66 = y_train.quantile(0.66)

bins = [-np.inf, q33, q66, np.inf]
labels = [0, 1, 2]

y_train_class = pd.cut(
    y_train,
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

y_test_class = pd.cut(
    y_test,
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

print("\nDistribution des classes de prix (train) :")
print(y_train_class.value_counts().sort_index().rename({
    0: "Bas",
    1: "Moyen",
    2: "Eleve"
}))

print("\nDistribution des classes de prix (test) :")
print(y_test_class.value_counts().sort_index().rename({
    0: "Bas",
    1: "Moyen",
    2: "Eleve"
}))

## 8. The Model Journey

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score
)

# ── Configuration des rounds ────────────────────────────────────────────────
ROUNDS = [
    {
        "label":  "Round 1 — Premier contact",
        "ridge":  Ridge(alpha=5.0),
        "forest": RandomForestRegressor(
            n_estimators=50, max_depth=4, min_samples_split=8,
            min_samples_leaf=4, random_state=42, n_jobs=1
        ),
        "boost":  GradientBoostingRegressor(
            n_estimators=50, max_depth=2, learning_rate=0.08,
            subsample=0.9, random_state=42
        ),
    },
    {
        "label":  "Round 2 — Ajustement de la profondeur",
        "ridge":  Ridge(alpha=1.0),
        "forest": RandomForestRegressor(
            n_estimators=100, max_depth=5, min_samples_split=6,
            min_samples_leaf=3, random_state=42, n_jobs=1
        ),
        "boost":  GradientBoostingRegressor(
            n_estimators=100, max_depth=2, learning_rate=0.06,
            subsample=0.9, random_state=42
        ),
    },
    {
        "label":  "Round 3 — Régularisation fine",
        "ridge":  Ridge(alpha=0.5),
        "forest": RandomForestRegressor(
            n_estimators=150, max_depth=6, min_samples_split=5,
            min_samples_leaf=2, random_state=42, n_jobs=1
        ),
        "boost":  GradientBoostingRegressor(
            n_estimators=150, max_depth=2, learning_rate=0.05,
            subsample=0.9, random_state=42
        ),
    },
    {
        "label":  "Round 4 — Réduction du biais",
        "ridge":  Ridge(alpha=0.1),
        "forest": RandomForestRegressor(
            n_estimators=200, max_depth=8, min_samples_split=4,
            min_samples_leaf=2, random_state=42, n_jobs=1
        ),
        "boost":  GradientBoostingRegressor(
            n_estimators=200, max_depth=3, learning_rate=0.04,
            subsample=0.85, random_state=42
        ),
    },
    {
        "label":  "Round 5 — Configuration finale optimisée",
        "ridge":  Ridge(alpha=0.05),
        "forest": RandomForestRegressor(
            n_estimators=300, max_depth=10, min_samples_split=4,
            min_samples_leaf=2, random_state=42, n_jobs=1
        ),
        "boost":  GradientBoostingRegressor(
            n_estimators=250, max_depth=3, learning_rate=0.03,
            subsample=0.85, random_state=42
        ),
    },
]

MODEL_NAMES = ["Ridge Regression", "Random Forest", "Gradient Boosting"]
COLORS      = ["#4C72B0", "#55A868", "#C44E52"]
MEDALS      = ["🥇", "🥈", "🥉"]

# ── Historique des métriques ────────────────────────────────────────────────
history = {
    name: {
        "MAE": [], "RMSE": [], "R2": [],
        "CV_R2_MEAN": [], "CV_R2_STD": [],
        "Accuracy": [], "Precision": [], "Recall": [],
        "SCORE_GLOBAL": []
    }
    for name in MODEL_NAMES
}

print("╔══════════════════════════════════════════════════════════╗")
print("║       THE MODEL JOURNEY — House Price Prediction         ║")
print("║       Version améliorée et plus robuste                  ║")
print("╚══════════════════════════════════════════════════════════╝\n")

# ── Boucle principale : calculs uniquement ─────────────────────────────────
for round_idx, round_cfg in enumerate(ROUNDS):
    round_num = round_idx + 1

    models_cfg = {
        "Ridge Regression":  (round_cfg["ridge"],  X_train_sc, X_test_sc, X_train_sc),
        "Random Forest":     (round_cfg["forest"], X_train,    X_test,    X_train),
        "Gradient Boosting": (round_cfg["boost"],  X_train,    X_test,    X_train),
    }

    round_scores = {}

    print(f"\n┌─ {round_cfg['label']} {'─' * max(1, 47 - len(round_cfg['label']))}")
    print(f"│  {'Modele':<22} {'MAE':>10} {'RMSE':>10} {'R2':>8} {'CV_R2':>10} {'Acc':>8}")
    print(f"│  {'─'*22} {'─'*10} {'─'*10} {'─'*8} {'─'*10} {'─'*8}")

    for name, (model, Xtr_fit, Xte_eval, Xtr_cv) in models_cfg.items():
        model_instance = clone(model)
        model_instance.fit(Xtr_fit, y_train)
        preds_reg = model_instance.predict(Xte_eval)

        mae  = mean_absolute_error(y_test, preds_reg)
        rmse = np.sqrt(mean_squared_error(y_test, preds_reg))
        r2   = r2_score(y_test, preds_reg)

        cv_model = clone(model)
        cv_scores = cross_val_score(
            cv_model,
            Xtr_cv,
            y_train,
            cv=5,
            scoring="r2",
            n_jobs=1
        )
        cv_r2_mean = cv_scores.mean()
        cv_r2_std  = cv_scores.std()

        preds_class = pd.cut(
            preds_reg,
            bins=bins,
            labels=labels,
            include_lowest=True
        ).astype(int)

        acc  = accuracy_score(y_test_class, preds_class)
        prec = precision_score(y_test_class, preds_class, average="weighted", zero_division=0)
        rec  = recall_score(y_test_class, preds_class, average="weighted", zero_division=0)

        score_global = (
            0.45 * r2 +
            0.35 * cv_r2_mean +
            0.10 * acc +
            0.05 * prec +
            0.05 * rec
        )

        history[name]["MAE"].append(mae)
        history[name]["RMSE"].append(rmse)
        history[name]["R2"].append(r2)
        history[name]["CV_R2_MEAN"].append(cv_r2_mean)
        history[name]["CV_R2_STD"].append(cv_r2_std)
        history[name]["Accuracy"].append(acc)
        history[name]["Precision"].append(prec)
        history[name]["Recall"].append(rec)
        history[name]["SCORE_GLOBAL"].append(score_global)

        round_scores[name] = score_global

        print(
            f"│  {name:<22} "
            f"{mae:>10,.0f} {rmse:>10,.0f} {r2:>8.4f} {cv_r2_mean:>10.4f} {acc:>8.4f}"
        )

    sorted_models = sorted(round_scores.items(), key=lambda x: x[1], reverse=True)

    print("│")
    for j, (m, score) in enumerate(sorted_models):
        print(f"│  {MEDALS[j]}  Rang {j + 1} : {m}  (Score global = {score:.4f})")

# ── Vainqueur final ─────────────────────────────────────────────────────────
final_scores = {name: history[name]["SCORE_GLOBAL"][-1] for name in MODEL_NAMES}
leader = max(final_scores, key=final_scores.get)

print("\n╔══════════════════════════════════════════════════════════╗")
print(f"║  {MEDALS[0]}  VAINQUEUR FINAL : {leader:<38}║")
print("╚══════════════════════════════════════════════════════════╝")

# ── Figure finale unique ────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor("#0f0f1a")
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)

ax_mae   = fig.add_subplot(gs[0, :2])
ax_r2    = fig.add_subplot(gs[1, :2])
ax_score = fig.add_subplot(gs[2, :2])
ax_rank  = fig.add_subplot(gs[:, 2])

for ax in [ax_mae, ax_r2, ax_score, ax_rank]:
    ax.set_facecolor("#1a1a2e")
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("#444")

rounds_x = list(range(1, len(ROUNDS) + 1))

# MAE
ax_mae.set_title("Evolution du MAE par round (plus bas = meilleur)", color="white", fontsize=10, pad=8)
for i, name in enumerate(MODEL_NAMES):
    ax_mae.plot(rounds_x, history[name]["MAE"], marker="o", color=COLORS[i], label=name, linewidth=2.5)
ax_mae.set_xlabel("Round", color="#aaa")
ax_mae.set_ylabel("MAE", color="#aaa")
ax_mae.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=8)
ax_mae.set_xticks(rounds_x)

# R2
ax_r2.set_title("Evolution du R2 test par round (plus haut = meilleur)", color="white", fontsize=10, pad=8)
for i, name in enumerate(MODEL_NAMES):
    ax_r2.plot(rounds_x, history[name]["R2"], marker="s", color=COLORS[i], label=name, linewidth=2.5)
ax_r2.set_xlabel("Round", color="#aaa")
ax_r2.set_ylabel("R2", color="#aaa")
ax_r2.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=8)
ax_r2.set_xticks(rounds_x)

# Score global
ax_score.set_title("Evolution du score global par round", color="white", fontsize=10, pad=8)
for i, name in enumerate(MODEL_NAMES):
    ax_score.plot(rounds_x, history[name]["SCORE_GLOBAL"], marker="D", color=COLORS[i], label=name, linewidth=2.5)
ax_score.set_xlabel("Round", color="#aaa")
ax_score.set_ylabel("Score global", color="#aaa")
ax_score.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=8)
ax_score.set_xticks(rounds_x)

# Classement final
sorted_final = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)
rank_names  = [m[0].replace(" Regression", "").replace(" Boosting", "") for m in sorted_final]
rank_scores = [m[1] for m in sorted_final]
rank_colors = [COLORS[MODEL_NAMES.index(m[0])] for m in sorted_final]

ax_rank.set_title("Classement final — Round 5", color="white", fontsize=10, pad=8)
bars = ax_rank.barh(rank_names, rank_scores, color=rank_colors, height=0.5)
ax_rank.set_xlim(0, max(1.0, max(rank_scores) + 0.1))
ax_rank.set_xlabel("Score global", color="#aaa")

for bar, score in zip(bars, rank_scores):
    ax_rank.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{score:.4f}",
        va="center",
        color="white",
        fontsize=9
    )

plt.suptitle(
    "THE MODEL JOURNEY — Résultats finaux",
    color="white",
    fontsize=13,
    fontweight="bold",
    y=0.98
)

plt.tight_layout()
plt.show()

## 9. Sélection du modèle vainqueur

In [ ]:
final_scores = {name: history[name]["R2"][-1] for name in MODEL_NAMES}
winner_name  = max(final_scores, key=final_scores.get)

winner_map = {
    "Ridge Regression":  (ROUNDS[-1]["ridge"],  X_train_sc, X_test_sc),
    "Random Forest":     (ROUNDS[-1]["forest"], X_train,    X_test),
    "Gradient Boosting": (ROUNDS[-1]["boost"],  X_train,    X_test),
}

best_model_obj, Xtr_w, Xte_w = winner_map[winner_name]
best_model_obj.fit(Xtr_w, y_train)
preds_final = best_model_obj.predict(Xte_w)

best_mae  = mean_absolute_error(y_test, preds_final)
best_rmse = np.sqrt(mean_squared_error(y_test, preds_final))
best_r2   = r2_score(y_test, preds_final)

preds_class_final = pd.cut(preds_final, bins=bins, labels=labels).astype(int)
best_acc  = accuracy_score(y_test_class,  preds_class_final)
best_prec = precision_score(y_test_class, preds_class_final,
                            average="weighted", zero_division=0)
best_rec  = recall_score(y_test_class,    preds_class_final,
                         average="weighted", zero_division=0)

print(f"Modele selectionne  : {winner_name}")
print(f"  MAE       : {best_mae:,.0f}")
print(f"  RMSE      : {best_rmse:,.0f}")
print(f"  R2        : {best_r2:.4f}")
print(f"  Accuracy  : {best_acc:.4f}")
print(f"  Precision : {best_prec:.4f}")
print(f"  Recall    : {best_rec:.4f}")

## 10. Optimisation avec GridSearch

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import GradientBoostingRegressor

print("GridSearch en cours sur Gradient Boosting...")
print("─" * 50)

param_grid = {
    "n_estimators":      [150, 200, 250, 300],
    "learning_rate":     [0.03, 0.05, 0.08],
    "max_depth":         [2, 3, 4],
    "subsample":         [0.85, 0.90, 1.0],
    "min_samples_split": [2, 5, 10]
}

grid_search = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring="r2",
    n_jobs=1,
    verbose=1,
    error_score="raise"
)

grid_search.fit(X_train, y_train)

print(f"\nMeilleurs parametres (GridSearch) :")
for k, v in grid_search.best_params_.items():
    print(f"  {k:<22} : {v}")

gs_preds = grid_search.best_estimator_.predict(X_test)
gs_r2    = r2_score(y_test, gs_preds)
gs_mae   = mean_absolute_error(y_test, gs_preds)
gs_rmse  = np.sqrt(mean_squared_error(y_test, gs_preds))

gs_preds_class = pd.cut(
    gs_preds,
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(int)

gs_acc  = accuracy_score(y_test_class, gs_preds_class)
gs_prec = precision_score(
    y_test_class,
    gs_preds_class,
    average="weighted",
    zero_division=0
)
gs_rec  = recall_score(
    y_test_class,
    gs_preds_class,
    average="weighted",
    zero_division=0
)

print(f"\nPerformances apres GridSearch :")
print(f"  MAE       : {gs_mae:,.0f}")
print(f"  RMSE      : {gs_rmse:,.0f}")
print(f"  R2        : {gs_r2:.4f}")
print(f"  Accuracy  : {gs_acc:.4f}")
print(f"  Precision : {gs_prec:.4f}")
print(f"  Recall    : {gs_rec:.4f}")

print(f"\n  {'Metrique':<12}  {'Avant':>8}  {'Apres':>8}  {'Gain':>8}")
print(f"  {'─'*12}  {'─'*8}  {'─'*8}  {'─'*8}")
print(f"  {'R2':<12}        {best_r2:>8.4f}  {gs_r2:>8.4f}  {gs_r2 - best_r2:>+8.4f}")
print(f"  {'MAE':<12}       {best_mae:>8,.0f}  {gs_mae:>8,.0f}  {gs_mae - best_mae:>+8,.0f}")
print(f"  {'Accuracy':<12}  {best_acc:>8.4f}  {gs_acc:>8.4f}  {gs_acc - best_acc:>+8.4f}")
print(f"  {'Precision':<12} {best_prec:>8.4f}  {gs_prec:>8.4f}  {gs_prec - best_prec:>+8.4f}")
print(f"  {'Recall':<12}    {best_rec:>8.4f}  {gs_rec:>8.4f}  {gs_rec - best_rec:>+8.4f}")

if gs_r2 >= best_r2:
    final_model      = grid_search.best_estimator_
    final_model_name = f"{winner_name} (GridSearch optimise)"
    final_metrics    = {
        "MAE": gs_mae,
        "RMSE": gs_rmse,
        "R2": gs_r2,
        "Accuracy": gs_acc,
        "Precision": gs_prec,
        "Recall": gs_rec
    }
    print(f"\nModele final retenu : GridSearch")
else:
    final_model      = best_model_obj
    final_model_name = winner_name
    final_metrics    = {
        "MAE": best_mae,
        "RMSE": best_rmse,
        "R2": best_r2,
        "Accuracy": best_acc,
        "Precision": best_prec,
        "Recall": best_rec
    }
    print(f"\nModele final retenu : Model Journey")

## 11. Enregistrement dans le Model Registry

In [ ]:
from snowflake.ml.registry import Registry

print("Connexion au Model Registry...")
print("─" * 50)

registry = Registry(session=session)

MODEL_NAME = "HOUSE_PRICE_MODEL"

print("\nEnregistrement du modele en cours...")

model_version = registry.log_model(
    final_model,
    model_name=MODEL_NAME,
    version_name="v1",
    
    sample_input_data=X_train.head(5),
    
    metrics={
        "MAE": final_metrics["MAE"],
        "RMSE": final_metrics["RMSE"],
        "R2": final_metrics["R2"],
        "Accuracy": final_metrics["Accuracy"],
        "Precision": final_metrics["Precision"],
        "Recall": final_metrics["Recall"],
    }
)

print("Modele enregistre avec succes")

## 12. Inférence depuis le Registry

In [ ]:
import pandas as pd

print("Chargement du modele depuis le registry...")
print("─" * 50)

MODEL_NAME = "HOUSE_PRICE_MODEL"

# Récupération du modèle versionné
model_ref = registry.get_model(MODEL_NAME)

# Selon ton workflow actuel, on utilise v1
prod_model = model_ref.version("v1")

print("Modele charge avec succes")

# ── Nouvelles données à scorer ──────────────────────────────────────────────
new_data = pd.DataFrame([{
    "AREA": 6000,
    "BEDROOMS": 3,
    "BATHROOMS": 2,
    "STORIES": 2,
    "PARKING": 1,
    "MAINROAD_ENC": 1,
    "GUESTROOM_ENC": 0,
    "BASEMENT_ENC": 1,
    "HOTWATER_ENC": 0,
    "AIRCON_ENC": 1,
    "PREFAREA_ENC": 1,
    "FURNISHING_ENC": 1
}])

print("\nDonnees a predire :")
print(new_data)

# Conversion Snowpark DataFrame
new_data_sf = session.create_dataframe(new_data)

# ── Inférence ───────────────────────────────────────────────────────────────
print("\nPrediction en cours...")
predictions = prod_model.run(new_data_sf, function_name="predict")

print("\nResultat de la prediction :")
predictions.show()

# ── Sauvegarde optionnelle ─────────────────────────────────────────────────
predictions.write.mode("overwrite").save_as_table("HOUSE_PRICE_PREDICTIONS")

print("\nPredictions sauvegardees dans la table HOUSE_PRICE_PREDICTIONS")

## 13. App Streamlit pour Snowflake

In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
import pandas as pd
import json

session = get_active_session()

st.set_page_config(page_title="House Price Estimator", layout="wide")

st.title("Estimation du prix d'une maison")
st.markdown("Renseignez les caractéristiques du bien pour obtenir une estimation de prix en temps réel.")
st.divider()

col1, col2, col3 = st.columns(3)

with col1:
    st.subheader("Superficie & Structure")
    area      = st.number_input("Surface (m2)", 30, 1000, 150, step=10)
    stories   = st.select_slider("Nombre d'étages", options=[1, 2, 3, 4], value=2)
    basement  = st.toggle("Sous-sol")
    mainroad  = st.toggle("Accès route principale")

with col2:
    st.subheader("Pièces & Confort")
    bedrooms  = st.select_slider("Chambres", options=[1, 2, 3, 4, 5, 6], value=3)
    bathrooms = st.select_slider("Salles de bain", options=[1, 2, 3, 4], value=2)
    parking   = st.select_slider("Places de parking", options=[0, 1, 2, 3], value=1)
    guestroom = st.toggle("Chambre d'amis")

with col3:
    st.subheader("Équipements & Localisation")
    aircon    = st.toggle("Climatisation")
    prefarea  = st.toggle("Zone privilégiée")
    furnishing = st.selectbox(
        "État d'ameublement",
        ["furnished", "semi-furnished", "unfurnished"],
        format_func=lambda x: {
            "furnished":      "Meublé",
            "semi-furnished": "Semi-meublé",
            "unfurnished":    "Non meublé"
        }[x]
    )

st.divider()

estimate = st.button("Estimer le prix", type="primary")

if estimate:
    enc      = lambda v: 1 if v else 0
    furn_map = {"furnished": 2, "semi-furnished": 1, "unfurnished": 0}

    query = f"""
        SELECT HOUSE_PRICE_DB.ML.HOUSE_PRICE_PREDICTOR!PREDICT(
            {int(area)}, {int(bedrooms)}, {int(bathrooms)},
            {int(stories)}, {int(parking)},
            {enc(mainroad)}, {enc(guestroom)}, {enc(basement)}, 0,
            {enc(aircon)}, {enc(prefarea)}, {furn_map[furnishing]}
        ) AS predicted_price
    """

    with st.spinner("Calcul en cours..."):
        result = session.sql(query).to_pandas()
        raw    = result.iloc[0, 0]

        if isinstance(raw, dict):
            price = float(list(raw.values())[0])
        elif isinstance(raw, str):
            try:
                parsed = json.loads(raw)
                price  = float(list(parsed.values())[0])
            except Exception:
                price  = float(raw)
        else:
            price = float(raw)

    st.divider()

    r1, r2, r3, r4 = st.columns(4)

    with r1:
        st.metric("Prix estimé", f"{price:,.0f} ₹")
    with r2:
        st.metric("Prix au m²", f"{price / area:,.0f} ₹")
    with r3:
        if price <= 250000:
            segment, delta = "Bas", "Marché accessible"
        elif price <= 350000:
            segment, delta = "Moyen", "Marché standard"
        else:
            segment, delta = "Élevé", "Marché premium"
        st.metric("Segment", segment, delta)
    with r4:
        st.metric("Budget avec marge 10%", f"{price * 1.10:,.0f} ₹")

    st.divider()
    st.subheader("Récapitulatif du bien")

    recap = pd.DataFrame({
        "Caractéristique": [
            "Surface", "Étages", "Chambres", "Salles de bain", "Parking",
            "Route principale", "Chambre d'amis", "Sous-sol",
            "Climatisation", "Zone privilégiée", "Ameublement"
        ],
        "Valeur": [
            f"{area} m²", stories, bedrooms, bathrooms, parking,
            "Oui" if mainroad  else "Non",
            "Oui" if guestroom else "Non",
            "Oui" if basement  else "Non",
            "Oui" if aircon    else "Non",
            "Oui" if prefarea  else "Non",
            {
                "furnished":      "Meublé",
                "semi-furnished": "Semi-meublé",
                "unfurnished":    "Non meublé"
            }[furnishing]
        ]
    })

    st.dataframe(recap, use_container_width=True, hide_index=True)